# Reusable Template — Asset Valuation Regression Pipeline

A dataset-agnostic notebook for **"predict a fair value/price from technical or condition attributes, optionally comparing against a market quote"** projects: aircraft/ship/equipment valuation, real-estate appraisal vs. asking price, used-vehicle pricing, industrial-equipment resale value, and similar problems.

**How to reuse this notebook:**
1. Fill in `CONFIG` (Section 1) for your dataset and any market-quote columns.
2. Fill in `clean_raw_columns()` (Section 3) with your dataset's unit-stripping / text-cleaning logic. **Check for the pandas `read_csv` "None"-as-NaN gotcha (Background Theory, Section 2) before trusting your null counts.**
3. Fill in `ORDINAL_MAPS` and `engineer_domain_features()` (Section 4) for any naturally-ordered condition categories and derived features.
4. Review the multiplicative-process check in Section 6 to decide whether to model the raw target or `log(target)`.
5. Review the leakage audit in Section 7 before finalizing your feature set.


## 1. Configuration

In [ ]:
CONFIG = {
    'data_path': 'aircraft_valuation.csv',      # <-- change per project
    'target_col': 'appraised_fair_market_value_usd',  # <-- change per project

    # Columns that are just another snapshot of the SAME value assessment you're
    # predicting -- exclude from features, keep in the DataFrame for later comparison.
    'market_quote_cols': ['broker_asking_price_usd', 'recent_comparable_sale_price_usd'],

    'id_cols': [],                        # columns to drop entirely
    'cardinality_threshold': 5,           # < threshold -> one-hot, >= threshold -> target-encode
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
}


## 2. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

RANDOM_STATE = CONFIG['random_state']

# Load defensively: if you know or suspect a genuine category might collide with pandas'
# default missing-value strings (e.g. "None", "NA"), set keep_default_na=False here and
# handle real missingness explicitly instead.
df_raw = pd.read_csv(CONFIG['data_path'])
df_raw = df_raw.drop(columns=[c for c in CONFIG['id_cols'] if c in df_raw.columns])
print(df_raw.shape)
df_raw.head()


## 3. Data Quality Audit (generic — reuse as-is)

In [ ]:
def audit_dataframe(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_unique': df.nunique(),
        'n_missing': df.isnull().sum(),
        'pct_missing': (df.isnull().mean() * 100).round(1),
        'sample': [df[c].dropna().unique()[:3].tolist() for c in df.columns],
    })

audit = audit_dataframe(df_raw)
print("Duplicate rows:", df_raw.duplicated().sum())
audit


In [ ]:
def check_for_na_string_collision(df, suspicious_defaults=('none', 'na', 'null', 'n/a')):
    """Flag columns where a suspiciously high null count might actually be a
    pandas default-NA-string collision (e.g. a real 'None' category being eaten).
    Not a fix -- just a nudge to go investigate before trusting the null count.
    """
    flags = []
    for c in df.columns:
        if df[c].isnull().mean() > 0.05:
            flags.append(c)
    if flags:
        print("Columns with >5% nulls -- verify these aren't a real category pandas ate:")
        print(flags)
    else:
        print("No columns flagged for manual null-count verification.")

check_for_na_string_collision(df_raw)


## 4. Cleaning & Feature Engineering (project-specific — EDIT THIS SECTION)

In [ ]:
# EDIT ME: naturally-ordered categories and their rank mappings (low -> high).
# Leave empty ({}) if your dataset has no ordinal categories.
ORDINAL_MAPS = {
    # 'condition_grade': {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3},
}


def clean_raw_columns(df):
    """EDIT ME. Strip units / parse combined-format text columns. Recover any
    genuine category that collided with pandas' default NA strings (see Section 3).
    Handle disguised-missing sentinels with the flag-column + impute pattern.
    Apply ORDINAL_MAPS to any naturally-ordered category columns.
    """
    df = df.copy()
    for col, mapping in ORDINAL_MAPS.items():
        if col in df.columns:
            df[col + '_score'] = df[col].map(mapping)
    # TODO: dataset-specific unit/text cleaning goes here
    return df


def engineer_domain_features(df):
    """EDIT ME. Ratio/derived features (e.g. age, utilization-per-unit-time,
    remaining-life-fraction) usually carry more signal than raw levels alone.
    """
    df = df.copy()
    # TODO: domain features go here
    return df


df = clean_raw_columns(df_raw)
df = engineer_domain_features(df)
df.head()


## 5. EDA (generic — reuse as-is)

In [ ]:
def eda_target_distribution(df, target_col):
    print(f"{target_col} raw skewness: {df[target_col].skew():.2f}")
    print(f"{target_col} log skewness: {np.log(df[target_col]).skew():.2f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[target_col], kde=True, ax=axes[0]).set_title(f'{target_col} (raw)')
    sns.histplot(np.log(df[target_col]), kde=True, ax=axes[1]).set_title(f'log({target_col})')
    plt.tight_layout(); plt.show()

eda_target_distribution(df, CONFIG['target_col'])


## 6. The Multiplicative-Process Check (generic — decide raw vs. log target)

If the log-skewness above is meaningfully lower than the raw skewness, and/or your target is strictly positive and spans multiple orders of magnitude, plan to compare a plain linear model against a log-linear one in Section 8 — don't assume one will obviously win without checking.

## 7. The Market-Quote Leakage Audit (generic — reuse as-is)

In [ ]:
def leakage_audit(df, target_col, known_market_cols, corr_threshold=0.9):
    numeric_df = df.select_dtypes(include='number')
    corrs = numeric_df.corr()[target_col].sort_values(ascending=False).drop(target_col)
    suspiciously_high = corrs[corrs.abs() >= corr_threshold]
    print("Top correlations with target:")
    print(corrs.head(10))
    print("\nFlagged as possible market-quote leakage (|corr| >=", corr_threshold, "):")
    for col, val in suspiciously_high.items():
        flagged = "already in market_quote_cols" if col in known_market_cols else "** NOT YET EXCLUDED -- REVIEW **"
        print(f"  {col:30s} corr={val:.3f}  ({flagged})")
    return corrs

_ = leakage_audit(df, CONFIG['target_col'], CONFIG['market_quote_cols'])


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(df, target_col, exclude_cols):
    numeric_df = df.select_dtypes(include='number')
    cols = [c for c in numeric_df.columns if c not in [target_col] + exclude_cols]
    X_vif = numeric_df[cols].dropna()
    return pd.DataFrame({
        'feature': X_vif.columns,
        'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    }).sort_values('VIF', ascending=False)

vif_table = compute_vif(df, CONFIG['target_col'], exclude_cols=CONFIG['market_quote_cols'])
vif_table.head(10)


## 8. Leakage-Safe Encoding (generic — reuse as-is)

In [ ]:
def split_columns_by_cardinality(df, threshold, exclude_cols):
    cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in exclude_cols]
    low_card = [c for c in cat_cols if df[c].nunique() < threshold]
    high_card = [c for c in cat_cols if df[c].nunique() >= threshold]
    return low_card, high_card


def prepare_features(df, config):
    # Exclude the ORIGINAL text version of any ordinal column (its *_score numeric
    # version, created in Section 4, is what actually goes into the model).
    ordinal_originals = list(ORDINAL_MAPS.keys())
    exclude_from_cat = [config['target_col']] + config['market_quote_cols'] + ordinal_originals
    low_card, high_card = split_columns_by_cardinality(df, config['cardinality_threshold'], exclude_from_cat)

    df_enc = df.drop(columns=[c for c in ordinal_originals if c in df.columns])
    df_enc = pd.get_dummies(df_enc, columns=low_card, drop_first=True)
    bool_cols = df_enc.select_dtypes(include='bool').columns
    df_enc[bool_cols] = df_enc[bool_cols].astype(int)

    drop_cols = [config['target_col']] + config['market_quote_cols']
    X = df_enc.drop(columns=[c for c in drop_cols if c in df_enc.columns])
    y = df_enc[config['target_col']]
    return X, y, low_card, high_card


from sklearn.model_selection import train_test_split

X, y, low_card, high_card = prepare_features(df, CONFIG)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG['test_size'], random_state=RANDOM_STATE
)

target_encoding_maps = {}
global_mean = y_train.mean()
for col in high_card:
    means = y_train.groupby(X_train[col]).mean()
    target_encoding_maps[col] = means
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(global_mean)

print("Nulls:", X_train.isnull().sum().sum(), X_test.isnull().sum().sum())
X_train.shape, X_test.shape


## 9. Model Zoo, Including the Log-Linear Comparison (generic — reuse as-is)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"{name:28s} MAE={mae:12,.2f}  RMSE={rmse:12,.2f}  R2={r2:.3f}")

y_pred_baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
evaluate('Mean baseline', y_test, y_pred_baseline)

lin = LinearRegression().fit(X_train, y_train)
evaluate('Linear Regression (raw)', y_test, lin.predict(X_test))

lin_log = LinearRegression().fit(X_train, np.log(y_train))
evaluate('Log-Linear Regression', y_test, np.exp(lin_log.predict(X_test)))

rf = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
evaluate('Random Forest', y_test, rf.predict(X_test))

gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
evaluate('Gradient Boosting', y_test, gb.predict(X_test))

fitted_models = {'Linear Regression (raw)': lin, 'Random Forest': rf, 'Gradient Boosting': gb}
pd.DataFrame(results).sort_values('MAE')


**Note:** if the log-linear model is competitive with or beats the raw-target models, prefer it (or a log-target tree model) for the rest of the pipeline when interpretability matters, even if a tree ensemble on the raw target scores marginally better.

## 10. Cross-Validation & Hyperparameter Tuning (generic — edit the grid)

In [ ]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

candidate_name = min(results[1:], key=lambda r: r['MAE'])['model']
candidate = fitted_models.get(candidate_name, gb)
print("Tuning:", candidate_name if candidate_name in fitted_models else 'Gradient Boosting (fallback)')

cv_scores = -cross_val_score(candidate, X_train, y_train, cv=CONFIG['cv_folds'],
                              scoring='neg_mean_absolute_error')
print(f"{CONFIG['cv_folds']}-fold CV MAE: {cv_scores.mean():,.2f} +/- {cv_scores.std():,.2f}")

param_dist = {'n_estimators': [100, 200, 400], 'max_depth': [2, 3, 4, 5],
              'learning_rate': [0.01, 0.05, 0.1, 0.2]} if hasattr(candidate, 'learning_rate') else {
    'n_estimators': [100, 200, 400], 'max_depth': [None, 5, 10, 20]
}
search = RandomizedSearchCV(candidate.__class__(random_state=RANDOM_STATE), param_distributions=param_dist,
                             n_iter=20, cv=CONFIG['cv_folds'], scoring='neg_mean_absolute_error',
                             random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
final_model = search.best_estimator_
evaluate('Final (tuned)', y_test, final_model.predict(X_test))


## 11. Diagnostics & Feature Importance (generic — reuse as-is)

In [ ]:
def plot_diagnostics(y_test, y_pred):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(y_test, y_pred, alpha=0.5)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[0].plot(lims, lims, 'r--'); axes[0].set_title('Predicted vs Actual')
    residuals = y_test - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.5); axes[1].axhline(0, color='r', linestyle='--')
    axes[1].set_title('Residuals vs Predicted')
    plt.tight_layout(); plt.show()

y_pred_final = final_model.predict(X_test)
plot_diagnostics(y_test, y_pred_final)


In [ ]:
from sklearn.inspection import permutation_importance

def plot_feature_importance(model, X_train, X_test, y_test, top_n=10):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        idx = np.argsort(importances)[-top_n:][::-1]
        sns.barplot(x=importances[idx], y=X_train.columns[idx], ax=axes[0])
        axes[0].set_title('Impurity-based importance')
    elif hasattr(model, 'coef_'):
        coefs = pd.Series(model.coef_, index=X_train.columns).sort_values(key=abs, ascending=False)
        sns.barplot(x=coefs.head(top_n).values, y=coefs.head(top_n).index, ax=axes[0])
        axes[0].set_title('|Coefficient| ranking')

    perm = permutation_importance(model, X_test, y_test, n_repeats=10,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    pidx = perm.importances_mean.argsort()[-top_n:][::-1]
    sns.barplot(x=perm.importances_mean[pidx], y=X_test.columns[pidx], ax=axes[1])
    axes[1].set_title('Permutation importance')
    plt.tight_layout(); plt.show()

plot_feature_importance(final_model, X_train, X_test, y_test)


## 12. Persistence & Generic Inference Wrapper

In [ ]:
import joblib

def save_artifacts(model, path_prefix='model'):
    joblib.dump(model, f'{path_prefix}.pkl')
    joblib.dump({
        'target_encoding_maps': target_encoding_maps,
        'global_mean': global_mean,
        'model_columns': list(X_train.columns),
        'low_card_cols': low_card,
        'high_card_cols': high_card,
        'ordinal_maps': ORDINAL_MAPS,
        'config': CONFIG,
    }, f'{path_prefix}_encoders.pkl')

def predict_from_raw(raw_dict, model, path_prefix='model'):
    art = joblib.load(f'{path_prefix}_encoders.pkl')
    row = pd.DataFrame([raw_dict])
    row = clean_raw_columns(row)
    row = engineer_domain_features(row)
    row = row.drop(columns=[c for c in art['ordinal_maps'].keys() if c in row.columns])
    row = pd.get_dummies(row, columns=art['low_card_cols'], drop_first=True)
    bool_cols = row.select_dtypes(include='bool').columns
    row[bool_cols] = row[bool_cols].astype(int)
    for col in art['high_card_cols']:
        if col in row.columns:
            row[col] = row[col].map(art['target_encoding_maps'][col]).fillna(art['global_mean'])
    row = row.reindex(columns=art['model_columns'], fill_value=0)
    return float(model.predict(row)[0])

save_artifacts(final_model)
print("Artifacts saved.")


## 13. Per-Project Checklist (quick reference)

- [ ] Update `CONFIG`, including `market_quote_cols`
- [ ] Check for a pandas default-NA-string collision before trusting null counts
- [ ] Fill in `ORDINAL_MAPS` for any naturally-ordered condition/quality categories
- [ ] Implement `clean_raw_columns()` and `engineer_domain_features()`
- [ ] Check target skewness raw vs. log; compare a log-linear model if the process looks multiplicative
- [ ] Run the leakage audit and review every flagged high-correlation column, reasoning about WHY each one is or isn't safe
- [ ] Confirm zero nulls in `X_train`/`X_test` after encoding
- [ ] Compare linear, log-linear, and tree-ensemble models — don't assume which wins
- [ ] Cross-validate before trusting a single split
- [ ] Choose metrics appropriate to the target's scale (MAPE is often good for strictly-positive, wide-range targets)
- [ ] Cross-check impurity/coefficient-based and permutation feature importance
- [ ] If comparing to a market quote, treat "settled transaction" columns and "opinion/quote" columns with different levels of leakage suspicion
- [ ] Persist encoders (including ordinal maps) alongside the model
